[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/para_casa/S12_07_autoencoder_generativos.ipynb)

# S12 · 07 — Autoencoders y modelos generativos (GAN y difusión)

🏠 Cuaderno para casa · ⏱ ≈ 50 min · Sprint 12 · UD 4.10

**Contexto TurisData.** A TurisData le interesa detectar **semanas atípicas** de ocupación (un evento, un fallo del sistema de reservas, una campaña) para investigarlas. No tenemos etiquetas de "anómalo": el autoencoder aprende lo **normal** y señala lo que no encaja.

**Este cuaderno tiene 5 ejercicios y un mini-reto** y es **solo para casa**: no tiene versión «en clase». Es ampliación: hazlo en casa, a tu ritmo y por orden.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, random, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); keras.utils.set_random_seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar la ocupación y convertirla en semanas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas semanas hay y qué forma tiene `semanas`?
2. ¿Qué representa cada fila?
3. ¿Por qué se usa error de reconstrucción alto como señal de anomalía?

In [ ]:
ocup = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")

serie = ocup["ocupacion_pct"]
n_semanas = len(serie) // 7
semanas = serie.values[:n_semanas * 7].reshape(n_semanas, 7)
fechas_semana = serie.index[:n_semanas * 7:7]
print("Semanas:", semanas.shape, "| primera semana empieza el", fechas_semana[0].date())

### ✏️ A3 · EJERCICIO 1 de 5 — Partir por orden temporal y escalar

**Tu turno:** Reserva el **último 20 %** de las semanas para prueba (orden temporal, sin barajar) y **escala** con `MinMaxScaler` ajustado solo con las semanas de entrenamiento. Ya tienes `corte` (dónde acaba el entrenamiento). Guarda el escalador en `escalador` y las matrices ya escaladas (7 columnas) en `sem_ent` y `sem_test`.

**Debe salir:** `Entrenamiento: (124, 7) | prueba: (32, 7)` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Cuántas semanas hay en entrenamiento y en prueba?
2. ¿Por qué no se baraja antes de partir?
3. ¿Qué error se evita ajustando el escalador solo con el entrenamiento?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
corte = int(0.8 * n_semanas)      # posición donde termina el entrenamiento y empieza la prueba
# pista: ajusta el escalador solo con las semanas anteriores a corte y transforma después los dos trozos
escalador = ...
sem_ent = ...
sem_test = ...

print("Entrenamiento:", sem_ent.shape, "| prueba:", sem_test.shape)
assert sem_ent.shape[1] == 7 and sem_test.shape[1] == 7, "Cada fila es una semana de 7 valores."
assert abs(sem_ent.min()) < 1e-9 and abs(sem_ent.max() - 1) < 1e-9, "El escalador se ajusta solo con entrenamiento: min 0 y max 1 en sem_ent."
assert len(sem_ent) + len(sem_test) == n_semanas, "No pierdas semanas al partir."
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Construir el autoencoder 7 → 4 → 2 → 4 → 7

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Cuántos parámetros tiene el autoencoder?
2. ¿Cuántas neuronas tiene el cuello de botella y para qué?
3. ¿Por qué la salida usa `sigmoid`?

In [ ]:
def construir_autoencoder(cuello=2):
    entrada = keras.Input(shape=(7,))
    x = layers.Dense(4, activation="relu")(entrada)
    z = layers.Dense(cuello, activation="relu", name="cuello")(x)
    x = layers.Dense(4, activation="relu")(z)
    salida = layers.Dense(7, activation="sigmoid")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
    return modelo

keras.utils.set_random_seed(SEMILLA)
ae = construir_autoencoder()
ae.summary(print_fn=lambda l: print(l))

### ✏️ A5 · EJERCICIO 2 de 5 — Entrenar el autoencoder

**Tu turno:** Entrena `ae` con `sem_ent` como **entrada y objetivo** (¡el mismo array dos veces!), 150 épocas, `batch_size=16`, `verbose=0`, con el 15 % final como validación (`validation_split=0.15`). Guarda el historial en `historial`.

**Debe salir:** `Error final de entrenamiento: 0.0076 | validación: 0.0095` (aproximado) y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Qué se pasa como entrada y como objetivo en `fit`?
2. ¿Qué error alcanza en entrenamiento y en validación?
3. ¿Qué indica que ambos errores sean parecidos?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: ae.fit(sem_ent, sem_ent, epochs=150, batch_size=16, validation_split=0.15, verbose=0)
historial = ...

print("Error final de entrenamiento:", round(historial.history["loss"][-1], 4), "| validación:", round(historial.history["val_loss"][-1], 4))
assert len(historial.history["loss"]) == 150, "Entrena 150 épocas."
assert historial.history["loss"][-1] < historial.history["loss"][0], "El error debería bajar con el entrenamiento; recuerda usar sem_ent como entrada Y como objetivo."
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Curva de error durante el entrenamiento

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué debería pasar con las dos curvas si el entrenamiento va bien?
2. ¿Qué indicaría una curva de validación que sube mientras la de entrenamiento baja?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(historial.history["loss"], label="entrenamiento"); ax.plot(historial.history["val_loss"], label="validación")
ax.set_title("Error de reconstrucción durante el entrenamiento"); ax.set_xlabel("Época"); ax.set_ylabel("Error cuadrático medio"); ax.legend()
plt.show()

### 👀 A7 · EJEMPLO — Comparar una semana real con su reconstrucción

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Para qué se usa `inverse_transform`?
2. ¿Cómo debe verse la reconstrucción de una semana normal?
3. ¿Por qué no coinciden exactamente?

In [ ]:
i = 5
original = sem_test[i:i + 1]
recons = ae.predict(original, verbose=0)
dias = ["dom", "lun", "mar", "mié", "jue", "vie", "sáb"]   # los datos empiezan el domingo 1/1/2023: cada semana de 7 días empieza en domingo
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(dias, escalador.inverse_transform(original)[0], "o-", label="Real")
ax.plot(dias, escalador.inverse_transform(recons)[0], "s--", label="Reconstruida")
ax.set_ylabel("Ocupación (%)"); ax.set_title("Una semana de prueba"); ax.legend()
plt.show()

### ✏️ A8 · EJERCICIO 3 de 5 — Error de reconstrucción y umbral

**Tu turno:** Completa `error_recons(modelo, X)`: devuelve un array con el error cuadrático medio de cada fila de `X` (media de `(X - modelo.predict(X))**2` por filas). Calcula `umbral` (percentil 95 de los errores de `sem_ent`, `np.percentile`).

**Debe salir:** `Umbral (percentil 95 en entrenamiento): 0.0154` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Qué umbral sale y cómo se calcula?
2. ¿Por qué se calcula con semanas de entrenamiento y no con todas?
3. ¿Qué consecuencia tiene usar el percentil 95?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
def error_recons(modelo, X):
    # pista: predice con el modelo y haz la media por filas del cuadrado de la diferencia (axis=1)
    ...

err_ent, err_test = error_recons(ae, sem_ent), error_recons(ae, sem_test)
umbral = float(np.percentile(err_ent, 95))
print(f"Umbral (percentil 95 en entrenamiento): {umbral:.4f}")
assert err_ent.shape == (len(sem_ent),), "error_recons debe devolver un error por semana (media por filas: axis=1)."
assert err_ent.min() >= 0, "Los errores cuadráticos no pueden ser negativos."
assert abs(np.mean(err_ent > umbral) - 0.05) < 0.03, "El umbral debe dejar ~5 % de las semanas de entrenamiento por encima."
print("✔ Ejercicio 3 correcto")

### 👀 A9 · EJEMPLO — Probar con anomalías inyectadas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Cuáles son las tres anomalías inyectadas?
2. ¿Cuál tiene el error más alto y cuánto?
3. ¿Por qué hay que fabricar anomalías para probar?

In [ ]:
normal = semanas[corte:corte + 3].copy()
raras = np.array([np.full(7, 15.0),                              # apagón
                  normal[0].copy(),                               # día roto
                  np.array([55., 52., 54., 53., 100., 55., 53.])])  # pico irreal
raras[1, 3] = 0.0
raras_esc = escalador.transform(raras)
err_raras = error_recons(ae, raras_esc)
for nombre, e in zip(["Apagón", "Día roto", "Pico irreal"], err_raras):
    print(f"{nombre:12s} error = {e:.4f}  →  {'ANOMALÍA' if e > umbral else 'normal'}")

### ✏️ A10 · EJERCICIO 4 de 5 — Anomalías detectadas y semanas reales marcadas

**Tu turno:** Calcula `n_detectadas` (cuántas de las 3 anomalías inyectadas superan el umbral) y `falsos_positivos` (cuántas semanas **reales de prueba**, `err_test`, superan el umbral).

**Debe salir:** `Anomalías detectadas: 3/3 | semanas reales marcadas: 3/32` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Cuántas anomalías detecta y cuántas semanas reales marca?
2. ¿Qué significa «semana real marcada»?
3. ¿Qué pasa con las alertas si se baja el percentil?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: suma de booleanos (err > umbral) en cada array
n_detectadas = ...
falsos_positivos = ...

print(f"Anomalías detectadas: {n_detectadas}/3 | semanas reales marcadas: {falsos_positivos}/{len(err_test)}")
assert 0 <= n_detectadas <= 3, "n_detectadas es un número entre 0 y 3."
assert n_detectadas >= 2, "El autoencoder debería detectar al menos 2 de las 3 anomalías inyectadas; revisa el umbral y el escalado."
assert falsos_positivos < len(err_test), "No todas las semanas reales pueden ser anómalas."
print("✔ Ejercicio 4 correcto")

### 👀 A11 · EJEMPLO — Error por semana frente al umbral

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Cuántos puntos superan la línea roja?
2. ¿Qué representa la línea roja?

In [ ]:
fig, ax = plt.subplots()
ax.plot(fechas_semana[corte:], err_test, "o-", ms=3, label="Semanas reales (prueba)")
ax.axhline(umbral, color="red", ls="--", label="Umbral (p95 entrenamiento)")
ax.set_title("Error de reconstrucción por semana (prueba)"); ax.set_ylabel("Error"); ax.set_xlabel("Fecha"); ax.legend()
plt.show()

### ✏️ A12 · MINI-RETO — ¿Qué semanas reales revisar?

**Tu turno:** Construye `revisar`: un DataFrame con las semanas de **prueba** cuyo error supera el umbral, con columnas `inicio_semana` (fecha) y `error`, ordenado de mayor a menor error. Si no hay ninguna, queda vacío (es una buena noticia). Escribe en `interpretacion` (texto) una frase para el cliente: qué significa que una semana aparezca ahí y qué **no** significa (no es prueba de fallo, solo "merece mirarse").

**Debe salir:** una tabla con 3 semanas (`2025-07-13` 0.0284, `2025-10-19` 0.0168, `2025-08-31` 0.0157) y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A12).

1. ¿Qué semana se revisaría primero y por qué?
2. ¿Qué le dirías al cliente de una semana que aparece en la lista?
3. ¿Qué harías si la lista tuviera cientos de semanas?

In [ ]:
# pista: DataFrame con fechas_semana[corte:] y err_test; filtra error > umbral; sort_values descendente
revisar = ...
interpretacion = "..."

print(revisar.head())
assert list(revisar.columns) == ["inicio_semana", "error"], "Columnas: inicio_semana y error."
assert (revisar["error"] > umbral).all(), "Solo semanas con error por encima del umbral."
assert revisar["error"].is_monotonic_decreasing, "Ordena de mayor a menor error."
assert isinstance(interpretacion, str) and len(interpretacion) > 40, "Escribe una interpretación en una o dos frases."
print("✔ Mini-reto correcto")

### ✏️ A13 · EJERCICIO 5 de 5 — Emparejar situaciones con la técnica adecuada

**Tu turno:** Asigna a `parejas` un diccionario que empareje cada situación con la técnica **más adecuada** (`"autoencoder"`, `"gan"` o `"difusion"`): - `"detectar semanas raras de ocupación"` - `"generar la foto de una terraza a partir de un texto"` - `"aprender a engañar a un discriminador con imágenes falsas"`

**Debe salir:** ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A13).

1. ¿Qué técnica detecta semanas raras sin etiquetas?
2. ¿Qué técnica genera una imagen a partir de un texto?
3. ¿Qué dos redes compiten en una GAN?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A13, **después de intentarlo**.

In [ ]:
# pista: sin etiquetas y reconstruyendo → autoencoder; imagen desde texto → difusión; competición generador/discriminador → GAN
parejas = ...

assert parejas.get("detectar semanas raras de ocupación") == "autoencoder", "Detectar rarezas sin etiquetas: autoencoder."
assert parejas.get("generar la foto de una terraza a partir de un texto") == "difusion", "Imagen desde texto: difusión."
assert parejas.get("aprender a engañar a un discriminador con imágenes falsas") == "gan", "Generador contra discriminador: GAN."
print("✔ Ejercicio 5 correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

**Contexto.** A TurisData le interesa detectar **semanas atípicas** de ocupación (un evento, un fallo del sistema de reservas, una campaña) para investigarlas. No tenemos etiquetas de «anómalo»: el autoencoder aprende lo **normal** y señala lo que no encaja.

- **Qué hace:**
    - Carga `numpy`, `pandas`, `matplotlib`, `keras` (`layers`) y `MinMaxScaler`; `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` reduce los mensajes de TensorFlow.
    - Fija todas las semillas (`random`, `numpy` y `keras`) para que los resultados se repitan, y un estilo común para los gráficos.
- **Qué debe salir:**
    - `Listo ✔` (la primera vez puede tardar unos segundos en cargar TensorFlow).
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Idea: comprimir y reconstruir

#### A2 · Ejemplo — Cargar la ocupación y convertirla en semanas

Un **autoencoder** tiene dos mitades:
- **Codificador:** reduce la entrada a pocos números (el **cuello de botella**, "resumen").
- **Decodificador:** intenta reconstruir la entrada original a partir de ese resumen.

Se entrena para que la salida sea **igual a la entrada**. Como el cuello es estrecho, la red solo puede guardar **lo esencial**:
los patrones habituales. Cuando le llega algo raro, lo reconstruye mal → **error de reconstrucción alto = posible anomalía**.

Vamos a representar cada **semana** (7 días de ocupación) como un vector de 7 números.

- **Qué hace:**
    - `pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")` carga la ocupación diaria con la fecha como índice (`ocup`).
    - `serie.values[:n_semanas * 7].reshape(n_semanas, 7)` agrupa los días en filas de 7: una fila por semana (`semanas`); `fechas_semana` guarda el día en que empieza cada una.
- **Qué debe salir:**
    - `Semanas: (156, 7) | primera semana empieza el 2023-01-01`.
- **Cómo leerlo:**
    - Hay **156 semanas** (tres años), cada una representada por un vector de 7 números. La primera empieza en domingo (1 de enero de 2023), así que todas las semanas empiezan en domingo.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas semanas hay y qué forma tiene `semanas`?* → **156** semanas; `semanas` es una matriz **(156, 7)**.
2. *¿Qué representa cada fila?* → Una **semana**: la ocupación (%) de sus 7 días.
3. *¿Por qué se usa error de reconstrucción alto como señal de anomalía?* → Porque el cuello estrecho solo deja guardar los **patrones habituales**: lo raro se reconstruye mal.

</details>

#### A3 · Ejercicio 1 — Partir por orden temporal y escalar

- **Qué hace:**
    - `corte = int(0.8 * n_semanas)` marca dónde acaba el entrenamiento (el 80 %) y empieza la prueba; no se baraja, porque es una serie temporal.
    - `MinMaxScaler().fit(semanas[:corte])` aprende el mínimo y el máximo **solo con el entrenamiento** (`escalador`).
    - `escalador.transform(...)` escala las semanas de entrenamiento (`sem_ent`) y de prueba (`sem_test`) con ese mismo escalador.
- **Qué debe salir:**
    - `Entrenamiento: (124, 7) | prueba: (32, 7)` y `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - 124 semanas para entrenar y las 32 últimas para probar, todas escaladas entre 0 y 1 con lo aprendido solo en el entrenamiento: así no se filtra información del futuro.
- **Si falla:**
    - `Cada fila es una semana de 7 valores.` → has aplanado las semanas o has escalado una sola columna.
    - `El escalador se ajusta solo con entrenamiento: min 0 y max 1 en sem_ent.` → has ajustado el escalador con todas las semanas (`semanas`) o con otro trozo.
    - `No pierdas semanas al partir.` → los cortes `[:corte]` y `[corte:]` no cubren todas las semanas.
    - `NameError: name 'semanas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas semanas hay en entrenamiento y en prueba?* → **124** y **32**.
2. *¿Por qué no se baraja antes de partir?* → Porque es una **serie temporal**: la prueba debe ser **posterior** al entrenamiento.
3. *¿Qué error se evita ajustando el escalador solo con el entrenamiento?* → La **fuga de datos**: que el mínimo y el máximo de la prueba influyan en el escalado.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
corte = int(0.8 * n_semanas)      # posición donde termina el entrenamiento y empieza la prueba
escalador = MinMaxScaler().fit(semanas[:corte])
sem_ent = escalador.transform(semanas[:corte])
sem_test = escalador.transform(semanas[corte:])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Construir y entrenar el autoencoder

#### A4 · Ejemplo — Construir el autoencoder 7 → 4 → 2 → 4 → 7

Arquitectura pequeña: 7 → 4 → **2** (cuello) → 4 → 7. Con solo 2 números tiene que resumir cada semana.

- **Qué hace:**
    - `construir_autoencoder(cuello=2)` define la red: entrada de 7 números, una capa de 4 neuronas, el **cuello de botella** de 2 neuronas (`cuello`), otra de 4 y una salida de 7 con `sigmoid` (los valores van entre 0 y 1).
    - Se compila con el optimizador `Adam` y pérdida `mse` (error cuadrático medio entre entrada y salida).
    - `ae.summary(...)` imprime la arquitectura.
- **Qué debe salir:**
    - Una tabla con las capas `input_layer (None, 7)`, `dense (None, 4)` con 32 parámetros, `cuello (None, 2)` con 10, `dense_1 (None, 4)` con 12 y `dense_2 (None, 7)` con 35, y `Total params: 89`.
- **Cómo leerlo:**
    - El cuello de **2 neuronas** obliga a resumir cada semana en solo dos números. La red es diminuta (89 parámetros), suficiente para aprender los patrones de una semana normal.
- **Si falla:**
    - `NameError: name 'keras' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros tiene el autoencoder?* → **89**.
2. *¿Cuántas neuronas tiene el cuello de botella y para qué?* → **2**: obliga a **resumir** cada semana y guardar solo lo esencial.
3. *¿Por qué la salida usa `sigmoid`?* → Porque los datos están escalados **entre 0 y 1** y la salida debe poder reconstruirlos.

</details>

#### A5 · Ejercicio 2 — Entrenar el autoencoder

- **Qué hace:**
    - `ae.fit(sem_ent, sem_ent, ...)` entrena la red con las semanas como **entrada y objetivo** a la vez: aprende a reproducir lo que recibe.
    - `epochs=150, batch_size=16` fijan el número de pasadas y el tamaño de lote; `validation_split=0.15` reserva el 15 % final para validar y `verbose=0` silencia el progreso.
    - El resultado (`historial`) guarda el error de cada época.
- **Qué debe salir:**
    - `Error final de entrenamiento: 0.0076 | validación: 0.0095` (los decimales pueden variar un poco entre ejecuciones) y `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Un error de unos 0,008 sobre datos escalados entre 0 y 1 es muy bajo: el autoencoder reconstruye bien las semanas normales. El de validación (0,0095) es algo mayor y parecido al de entrenamiento: no hay sobreajuste importante.
- **Si falla:**
    - `Entrena 150 épocas.` → `epochs` no es 150.
    - `El error debería bajar con el entrenamiento; recuerda usar sem_ent como entrada Y como objetivo.` → has pasado otro objetivo en lugar de `sem_ent` (tiene que ser el mismo array dos veces).
    - `NameError: name 'ae' is not defined` → falta el ejemplo que construye el autoencoder.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué se pasa como entrada y como objetivo en `fit`?* → **El mismo array** (`sem_ent`) dos veces: la red aprende a reproducir lo que recibe.
2. *¿Qué error alcanza en entrenamiento y en validación?* → Alrededor de **0,008** y **0,010**.
3. *¿Qué indica que ambos errores sean parecidos?* → Que **no hay un sobreajuste importante**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
historial = ae.fit(sem_ent, sem_ent, epochs=150, batch_size=16, validation_split=0.15, verbose=0)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — Curva de error durante el entrenamiento

- **Qué hace:**
    - Dibuja el error de entrenamiento (`loss`) y de validación (`val_loss`) de cada época a partir del `historial`.
- **Qué debe salir:**
    - Un gráfico «Error de reconstrucción durante el entrenamiento» con dos curvas que bajan con las épocas.
- **Cómo leerlo:**
    - Las dos curvas bajan y se mantienen cercanas: el autoencoder está aprendiendo a reconstruir también semanas que no ha visto en el entrenamiento (validación).
- **Si falla:**
    - `NameError: name 'historial' is not defined` → falta el ejercicio 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué debería pasar con las dos curvas si el entrenamiento va bien?* → **Bajar** y mantenerse **cercanas** entre sí.
2. *¿Qué indicaría una curva de validación que sube mientras la de entrenamiento baja?* → **Sobreajuste**: memoriza el entrenamiento y generaliza peor.

</details>

## 3. Ver una reconstrucción

#### A7 · Ejemplo — Comparar una semana real con su reconstrucción

Comparamos una semana real con la que reconstruye el autoencoder (deshaciendo el escalado).

- **Qué hace:**
    - `sem_test[i:i + 1]` toma una semana de prueba (la nº 5) y `ae.predict(...)` la reconstruye.
    - `escalador.inverse_transform(...)` deshace el escalado para ver porcentajes de ocupación, y se dibujan la semana real (círculos) y la reconstruida (cuadrados) por días (de domingo a sábado).
- **Qué debe salir:**
    - Un gráfico «Una semana de prueba» con dos líneas, «Real» y «Reconstruida», sobre los días dom–sáb.
- **Cómo leerlo:**
    - Las dos líneas son parecidas: para una semana normal, el autoencoder reproduce bastante bien la forma del patrón semanal (aunque no exactamente cada día). Es lo que se espera de una semana «normal».
- **Si falla:**
    - `NameError: name 'sem_test' is not defined` → falta el ejercicio 1.
    - `NameError: name 'ae' is not defined` → falta el ejemplo que construye el autoencoder.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Para qué se usa `inverse_transform`?* → Para **deshacer el escalado** y ver la ocupación en porcentaje.
2. *¿Cómo debe verse la reconstrucción de una semana normal?* → **Parecida** a la real: sigue su forma general.
3. *¿Por qué no coinciden exactamente?* → Porque el cuello de 2 neuronas **solo guarda lo esencial**, no cada detalle.

</details>

## 4. Detección de anomalías

#### A8 · Ejercicio 3 — Error de reconstrucción y umbral

1. Calculamos el **error de reconstrucción** de cada semana (error cuadrático medio de sus 7 días).
2. Fijamos un **umbral** con las semanas de **entrenamiento** (normales): por ejemplo, el **percentil 95** de sus errores.
3. Una semana con error por encima del umbral se marca como sospechosa.

- **Qué hace:**
    - `modelo.predict(X, verbose=0)` reconstruye cada semana.
    - `np.mean((X - pred) ** 2, axis=1)` calcula, para cada fila (semana), la media del error cuadrático de sus 7 días: un número por semana.
    - La comprobación calcula los errores de entrenamiento y prueba y fija `umbral` con `np.percentile(err_ent, 95)`: el valor por debajo del cual está el 95 % de las semanas normales de entrenamiento.
- **Qué debe salir:**
    - `Umbral (percentil 95 en entrenamiento): 0.0154` y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Una semana cuyo error supere 0,0154 se considera sospechosa. Por construcción, cerca de un **5 %** de las semanas de entrenamiento queda por encima: es el precio de usar el percentil 95.
- **Si falla:**
    - `error_recons debe devolver un error por semana (media por filas: axis=1).` → has hecho la media de toda la matriz o por columnas (usa `axis=1`).
    - `Los errores cuadráticos no pueden ser negativos.` → has restado sin elevar al cuadrado.
    - `El umbral debe dejar ~5 % de las semanas de entrenamiento por encima.` → los errores no están bien calculados.
    - `NameError: name 'sem_ent' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué umbral sale y cómo se calcula?* → **0,0154**: el percentil 95 de los errores de las semanas de **entrenamiento**.
2. *¿Por qué se calcula con semanas de entrenamiento y no con todas?* → Porque son las semanas **normales** que conoce el modelo; las raras no deben influir en el umbral.
3. *¿Qué consecuencia tiene usar el percentil 95?* → Que alrededor del **5 %** de las semanas normales saldrá marcado como sospechoso.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def error_recons(modelo, X):
    pred = modelo.predict(X, verbose=0)
    return np.mean((X - pred) ** 2, axis=1)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A9 · Ejemplo — Probar con anomalías inyectadas

No sabemos qué semanas reales son "raras", así que **fabricamos** casos de prueba para ver si el modelo los detecta:
- Un **apagón de reservas**: una semana con ocupación baja todos los días (p. ej. 15 %).
- Un **día roto**: una semana normal pero con un día a 0 % (fallo del sistema).
- Un **pico irreal**: un día al 100 % en una semana floja.
Se escala igual que el resto de los datos.

- **Qué hace:**
    - `raras` contiene tres semanas inventadas: un **apagón** (15 % todos los días), un **día roto** (una semana normal con un día a 0 %) y un **pico irreal** (un día al 100 % en una semana floja).
    - `escalador.transform(raras)` las escala igual que el resto y `error_recons` calcula su error.
    - El bucle imprime el error de cada una y si supera el `umbral` (`ANOMALÍA`) o no (`normal`).
- **Qué debe salir:**
    - `Apagón error = 1.4394 → ANOMALÍA`, `Día roto error = 0.2663 → ANOMALÍA` y `Pico irreal error = 0.2401 → ANOMALÍA`.
- **Cómo leerlo:**
    - Los tres errores son **muy superiores al umbral** (0,0154): el apagón es 90 veces mayor. El modelo, que solo ha visto semanas normales, reconstruye muy mal lo que no encaja.
- **Si falla:**
    - `NameError: name 'umbral' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuáles son las tres anomalías inyectadas?* → Un **apagón** de reservas, un **día roto** (a 0 %) y un **pico irreal** (un día al 100 %).
2. *¿Cuál tiene el error más alto y cuánto?* → El **apagón**, con **1,44**.
3. *¿Por qué hay que fabricar anomalías para probar?* → Porque **no tenemos etiquetas** de semanas raras reales: así se comprueba que el modelo las detecta.

</details>

#### A10 · Ejercicio 4 — Anomalías detectadas y semanas reales marcadas

- **Qué hace:**
    - `np.sum(err_raras > umbral)` cuenta cuántas de las 3 anomalías inyectadas superan el umbral (`n_detectadas`).
    - `np.sum(err_test > umbral)` cuenta cuántas semanas **reales** de prueba lo superan (`falsos_positivos`); `int(...)` los convierte en enteros normales.
- **Qué debe salir:**
    - `Anomalías detectadas: 3/3 | semanas reales marcadas: 3/32` y `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - Detecta **las 3** anomalías inyectadas y marca como sospechosas **3 de las 32** semanas reales de prueba (≈ 9 %). No sabemos si esas 3 son realmente raras: merecen mirarse. El 5 % de «falsos avisos» esperado en entrenamiento aquí sube algo.
- **Si falla:**
    - `n_detectadas es un número entre 0 y 3.` → has contado sobre otro array.
    - `El autoencoder debería detectar al menos 2 de las 3 anomalías inyectadas; revisa el umbral y el escalado.` → has comparado con otro umbral o con `err_test` en lugar de `err_raras`.
    - `No todas las semanas reales pueden ser anómalas.` → has comparado `err_ent`/`err_raras` en lugar de `err_test` en la segunda cuenta.
    - `NameError: name 'err_raras' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas anomalías detecta y cuántas semanas reales marca?* → **3 de 3** anomalías y **3 de 32** semanas reales.
2. *¿Qué significa «semana real marcada»?* → Que su error supera el umbral: **merece revisarse**, no es prueba de fallo.
3. *¿Qué pasa con las alertas si se baja el percentil?* → Habría **más alertas** (más falsos avisos) pero se escaparían menos anomalías.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
n_detectadas = int(np.sum(err_raras > umbral))
falsos_positivos = int(np.sum(err_test > umbral))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A11 · Ejemplo — Error por semana frente al umbral

- **Qué hace:**
    - Dibuja el error de reconstrucción de cada semana de prueba a lo largo del tiempo y una línea roja discontinua con el `umbral`.
- **Qué debe salir:**
    - Un gráfico «Error de reconstrucción por semana (prueba)» con puntos unidos y la línea roja del umbral (p95 entrenamiento).
- **Cómo leerlo:**
    - Casi todos los puntos quedan por debajo de la línea; tres la superan. Esos son los candidatos a revisar.
- **Si falla:**
    - `NameError: name 'err_test' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos puntos superan la línea roja?* → **Tres** semanas de prueba.
2. *¿Qué representa la línea roja?* → El **umbral** (percentil 95 del error en entrenamiento).

</details>

## 5. Mini-reto: ¿qué semanas reales revisar?

#### A12 · Mini-reto — ¿Qué semanas reales revisar?

- **Qué hace:**
    - `pd.DataFrame({"inicio_semana": fechas_semana[corte:], "error": err_test})` junta la fecha de inicio y el error de cada semana de prueba.
    - `revisar["error"] > umbral` deja solo las semanas sospechosas; `sort_values("error", ascending=False)` las ordena de mayor a menor error y `reset_index(drop=True)` renumera las filas.
    - `interpretacion` es la frase para el cliente: qué significa que una semana aparezca (patrón distinto al habitual) y qué **no** significa (no es prueba de fallo).
- **Qué debe salir:**
    - Una tabla `inicio_semana`/`error` con 3 filas: `2025-07-13` (0.028356), `2025-10-19` (0.016830) y `2025-08-31` (0.015676), y `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - La primera semana de la lista (13 de julio de 2025) es la que peor reconstruye el modelo: es la primera que habría que mirar. Pero **aparecer en la lista no prueba un fallo**: solo indica que su patrón se aleja de lo habitual (un evento, una promoción, un fallo de datos…).
- **Si falla:**
    - `Columnas: inicio_semana y error.` → las columnas tienen otro nombre u otro orden.
    - `Solo semanas con error por encima del umbral.` → falta el filtro `error > umbral`.
    - `Ordena de mayor a menor error.` → falta `ascending=False`.
    - `Escribe una interpretación en una o dos frases.` → `interpretacion` está vacía o es muy corta (más de 40 caracteres).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué semana se revisaría primero y por qué?* → La del **13 de julio de 2025**: es la de **mayor error** (0,028).
2. *¿Qué le dirías al cliente de una semana que aparece en la lista?* → Que su patrón es **distinto al habitual** y merece mirarse, pero **no es prueba de que algo haya fallado**.
3. *¿Qué harías si la lista tuviera cientos de semanas?* → **Subir el umbral** (por ejemplo, el percentil 99) para que solo salgan las más extremas.

</details>

## 6. Otros modelos generativos: ideas (sin entrenar)

#### A13 · Ejercicio 5 — Emparejar situaciones con la técnica adecuada

El autoencoder **comprime y reconstruye**. Los modelos **generativos** dan un paso más: **crean datos nuevos** parecidos a los de entrenamiento.
No los entrenamos aquí (necesitan GPU y mucho tiempo), pero conviene entender la idea.

### GAN (*Generative Adversarial Network*): el falsificador y el policía
```
  ruido aleatorio ──► [ GENERADOR ] ──► imagen falsa ─┐
                                                       ├──► [ DISCRIMINADOR ] ──► ¿real o falsa?
                         imágenes reales ─────────────┘            │
       ▲                                                             │
       └───────────── ambos se corrigen con el resultado ◄──────────┘
```
- El **generador** intenta fabricar ejemplos tan buenos que engañen al discriminador.
- El **discriminador** intenta distinguir lo real de lo falso.
- Al competir, ambos mejoran. Resultado: imágenes muy realistas (fotos de habitaciones, caras que no existen).
- **Problemas típicos:** entrenamiento inestable; el generador repite pocos ejemplos (*mode collapse*).

### Modelos de difusión: de la niebla a la imagen
```
  ENTRENAMIENTO (se aprende a quitar ruido):
  imagen ──► +ruido ──► +ruido ──► ... ──► puro ruido       (proceso "hacia delante", fijo)

  GENERACIÓN (se aplica al revés):
  puro ruido ──► red quita un poco ──► ... ──► imagen nueva  (guiada por el texto: "una terraza al atardecer en Tenerife")
```
- Se entrena una red para **quitar un poquito de ruido** en cada paso.
- Para crear, se parte de ruido puro y se limpia paso a paso, guiando el resultado con un texto.
- Estables de entrenar y de gran calidad; son la base de muchos generadores de imágenes actuales. Más lentos que una GAN al generar.

### Comparación rápida
| | Autoencoder | GAN | Difusión |
|---|---|---|---|
| Idea | comprimir y reconstruir | competir (falsificador vs. policía) | quitar ruido paso a paso |
| Uso típico | anomalías, compresión | imágenes realistas | imágenes y vídeo guiados por texto |
| Ventaja | simple, rápido | resultados nítidos | calidad y estabilidad |
| Inconveniente | no genera cosas nuevas por sí solo | inestable | lento al generar, caro |

- **Qué hace:**
    - `parejas` es un diccionario que asigna a cada situación una de las tres técnicas: `"autoencoder"`, `"gan"` o `"difusion"`.
    - Las tres situaciones: detectar semanas raras (sin etiquetas, reconstruyendo → autoencoder), generar una foto desde un texto (quitando ruido paso a paso → difusión) y aprender a engañar a un discriminador (generador contra discriminador → GAN).
- **Qué debe salir:**
    - `✔ Ejercicio 5 correcto`.
- **Cómo leerlo:**
    - Cada técnica tiene su idea: el **autoencoder** comprime y reconstruye, la **GAN** compite (falsificador y policía) y la **difusión** limpia ruido paso a paso, guiada por texto.
- **Si falla:**
    - `Detectar rarezas sin etiquetas: autoencoder.` → has escrito otra técnica, o con otro texto (usa exactamente `"autoencoder"`, `"gan"` o `"difusion"`, sin tilde).
    - `Imagen desde texto: difusión.` → la clave `"difusion"` va sin tilde y en minúsculas.
    - `Generador contra discriminador: GAN.` → comprueba que el texto de la situación coincide exactamente con el del enunciado.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué técnica detecta semanas raras sin etiquetas?* → El **autoencoder**.
2. *¿Qué técnica genera una imagen a partir de un texto?* → Los modelos de **difusión**.
3. *¿Qué dos redes compiten en una GAN?* → El **generador** (falsificador) y el **discriminador** (policía).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
parejas = {"detectar semanas raras de ocupación": "autoencoder",
           "generar la foto de una terraza a partir de un texto": "difusion",
           "aprender a engañar a un discriminador con imágenes falsas": "gan"}
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 7. IA generativa responsable
Si TurisData usara imágenes o textos generados en su web o sus anuncios:
- **Veracidad:** una foto sintética de una habitación que no existe puede ser **publicidad engañosa**. Hay que etiquetar el contenido generado y no usarlo para mostrar el alojamiento real.
- **Derechos:** ¿con qué datos se entrenó el modelo? ¿Quién es el autor de lo generado? Revisa licencias.
- **Sesgos:** los modelos reproducen estereotipos de sus datos (¿cómo se representa al "turista" o al "personal"?).
- **Datos personales:** no subir fotos de huéspedes ni reseñas con nombres a servicios externos.
- **Supervisión humana:** una persona revisa antes de publicar.

## Preguntas de reflexión
1. ¿Por qué el cuello de botella tiene que ser **estrecho**? ¿Qué pasaría con un cuello de 7 neuronas (igual que la entrada)?
2. El umbral del percentil 95 marca ~5 % de las semanas normales como anómalas. ¿Cómo lo cambiarías si revisar una alerta cuesta mucho tiempo?
3. ¿Qué otras variables (temperatura, festivos, precio) podrías incluir en el vector de cada semana?
4. Escribe una norma de uso de imágenes generadas por IA para el equipo de marketing de TurisData (3 líneas).

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Porque el cuello estrecho **obliga a resumir**: solo cabe lo esencial (los patrones habituales), y lo raro se reconstruye mal, que es lo que se usa para detectar anomalías. Con un cuello de 7 neuronas la red podría copiar la entrada tal cual (reconstruiría **todo** bien, incluso lo raro) y el error ya no serviría como señal.
2. **Subiendo el percentil** (por ejemplo, al 99): salen menos alertas, aunque se pueden escapar anomalías más suaves. Es una decisión de coste: cuánto cuesta revisar una alerta frente a cuánto cuesta no ver una anomalía real.
3. Variables que explican la ocupación: **temperatura**, **festivos y eventos**, **precio**, día de la semana, campañas o reservas anticipadas. Así el modelo sabría que una semana de festivos es normal y no la marcaría.
4. Por ejemplo: «Las imágenes generadas por IA se etiquetan siempre como tales. No se usan para mostrar el alojamiento real ni personas reales. Una persona revisa cada imagen antes de publicarla (derechos, sesgos y datos personales).»

</details>

## Resumen: qué has aprendido
- Un **autoencoder** aprende a comprimir y reconstruir lo normal; el **error de reconstrucción** señala lo raro (anomalías sin etiquetas).
- El umbral se fija con datos normales de **entrenamiento** y se prueba con anomalías inyectadas y semanas reales.
- **GAN** = competición entre generador y discriminador; **difusión** = quitar ruido paso a paso, guiado por texto.
- La IA generativa exige criterios de veracidad, derechos, sesgos y supervisión humana.